# PIDA-RL Diabetes — 03. Preparación reproducible de cohorte

Este notebook construye la cohorte analítica de personas adultas con diabetes diagnosticada a partir de ENSANUT Continua 2022. Usa `FOLIO_INT` como llave única, `a0301 = 1` como criterio de inclusión y excluye explícitamente `a0301 = 2` (diabetes gestacional).

Las decisiones de codificación documentadas son: variables binarias `Sí = 1`, `No = 2`, código especial `9`; variables continuas con códigos especiales `89` y `99`; sexo `Hombre = 1`, `Mujer = 2`.

> Alcance: esta cohorte y su score se utilizan para un simulador de priorización de seguimiento. No constituyen una escala clínica validada ni una recomendación individual de atención.

## 1. Inicialización robusta

El notebook detecta Drive en `/content/drive` o `/content/gdrive`, crea las carpetas persistentes y recupera el repositorio temporal si la sesión de Colab se reinició.

In [1]:
from pathlib import Path
import os
import sys
import json
import subprocess
from datetime import datetime, timezone

PROJECT_NAME = 'pida-rl-datos-publicos'
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
PROJECT_ROOT = Path('/content') / PROJECT_NAME

DRIVE_ROOT = next(
    (path for path in [Path('/content/drive'), Path('/content/gdrive')]
     if (path / 'MyDrive').is_dir()),
    None
)

if DRIVE_ROOT is None:
    from google.colab import drive
    mount_point = Path('/content/drive')
    if mount_point.exists() and any(mount_point.iterdir()):
        mount_point = Path('/content/gdrive')
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError(
            f'No es posible montar Drive en {mount_point}; contiene archivos locales. '
            'Reinicia la sesión de Colab o usa un punto de montaje vacío.'
        )
    drive.mount(str(mount_point), force_remount=False)
    DRIVE_ROOT = mount_point

MYDRIVE_ROOT = DRIVE_ROOT / 'MyDrive'
PERSISTENT_ROOT = MYDRIVE_ROOT / 'PIDA-RL-Diabetes'
DATA_ROOT = PERSISTENT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'

for directory in [RAW_DATA_DIR, PROCESSED_DATA_DIR, METADATA_DIR, SPECIFICATIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'La ruta no es un repositorio Git: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
SRC_ROOT = PROJECT_ROOT / 'src'
SRC_ROOT.mkdir(parents=True, exist_ok=True)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('Proyecto:', PROJECT_ROOT)
print('Drive:', DRIVE_ROOT)
print('Datos crudos:', RAW_DATA_DIR)
print('Datos procesados:', PROCESSED_DATA_DIR)


Mounted at /content/drive
Proyecto: /content/pida-rl-datos-publicos
Drive: /content/drive
Datos crudos: /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw
Datos procesados: /content/drive/MyDrive/PIDA-RL-Diabetes/data/processed


## 2. Cargar Adultos ENSANUT

Reutiliza `df_adultos` si viene del notebook 01. Si se abre una sesión nueva, localiza el archivo `.sav` de Adultos dentro de `data/raw`.

In [2]:
import numpy as np
import pandas as pd

try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyreadstat'])
    import pyreadstat

def find_adult_source(raw_dir):
    files = sorted(path for path in raw_dir.rglob('*.sav') if path.is_file())
    expected_names = ['AdultosENSANUT2022.sav', 'Adultos_ENSANUT_2022.sav']
    for name in expected_names:
        exact = [path for path in files if path.name.lower() == name.lower()]
        if exact:
            return exact[0]
    matches = [path for path in files if 'adult' in path.name.lower() and 'ensanut' in path.name.lower()]
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        f'No se encontró el archivo Adultos ENSANUT en {raw_dir}. '
        'Ejecuta 01_fuentes_y_carga_reinicio.ipynb o coloca el archivo .sav en data/raw.'
    )

if 'df_adultos' not in globals():
    ADULTOS_PATH = find_adult_source(RAW_DATA_DIR)
    df_adultos, meta_adultos = pyreadstat.read_sav(ADULTOS_PATH, apply_value_formats=False)
    print('Fuente recargada:', ADULTOS_PATH)
else:
    ADULTOS_PATH = None
    print('Se reutiliza df_adultos de la sesión actual.')

print('Dimensiones de Adultos:', df_adultos.shape)


Fuente recargada: /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw/Adultos_ENSANUT_2022.sav
Dimensiones de Adultos: (11913, 572)


## 3. Especificación cerrada de variables y códigos

Estas codificaciones fueron verificadas durante el entendimiento de datos y se aplican de manera consistente en todo el notebook.

In [3]:
IDENTIFIER = 'FOLIO_INT'
AGE_VAR = 'edad'
SEX_VAR = 'sexo'
DIABETES_VAR = 'a0301'
AGE_AT_DIAGNOSIS_VAR = 'a0302'
INSULIN_VAR = 'a0310'
DEPRESSION_VAR = 'a0202'
CARE_VISITS_VAR = 'a0303num'
HOSPITALIZATION_VAR = 'a03061b'

COMPLICATION_VARS = {
    'ulceras_piernas_pies': 'a0316a',
    'amputacion': 'a0316b',
    'disminucion_vision': 'a0316c',
    'perdida_vista': 'a0316d',
    'dialisis': 'a0316e',
    'infarto_cardiaco': 'a0316f',
    'infarto_cerebral': 'a0316g',
    'pie_diabetico': 'a0316h',
    'coma_diabetico': 'a0316i',
    'hipoglucemia_grave': 'a0316j'
}

# Codificación confirmada por el usuario para ENSANUT.
DIABETES_POSITIVE_VALUES = [1]
GESTATIONAL_DIABETES_VALUES = [2]
DIABETES_NEGATIVE_VALUES = [3]
BINARY_YES_VALUES = [1]
BINARY_NO_VALUES = [2]
BINARY_SPECIAL_MISSING_CODES = [9]
CONTINUOUS_SPECIAL_MISSING_CODES = [89, 99]
SEX_CODES = {1: 'hombre', 2: 'mujer'}

REQUIRED_COLUMNS = [
    IDENTIFIER, AGE_VAR, SEX_VAR, DIABETES_VAR, AGE_AT_DIAGNOSIS_VAR,
    INSULIN_VAR, DEPRESSION_VAR, CARE_VISITS_VAR, HOSPITALIZATION_VAR,
    *COMPLICATION_VARS.values()
]

missing_columns = [column for column in REQUIRED_COLUMNS if column not in df_adultos.columns]
if missing_columns:
    raise KeyError('Faltan columnas requeridas: ' + ', '.join(missing_columns))

print('Columnas requeridas verificadas:', len(REQUIRED_COLUMNS))
print('Códigos de diagnóstico: Sí=1, Gestacional=2, No=3')
print('Binarias: Sí=1, No=2, especial=9')
print('Continuas: especiales=89, 99')
print('Sexo: 1=hombre, 2=mujer')


Columnas requeridas verificadas: 19
Códigos de diagnóstico: Sí=1, Gestacional=2, No=3
Binarias: Sí=1, No=2, especial=9
Continuas: especiales=89, 99
Sexo: 1=hombre, 2=mujer


## 4. Auditoría previa de codificaciones

La celda muestra distribuciones antes del filtrado. Los registros con `a0301 = 2` se excluirán explícitamente como diabetes gestacional.

In [4]:
print('Distribución de diagnóstico de diabetes (a0301):')
display(df_adultos[DIABETES_VAR].value_counts(dropna=False).sort_index())

print('Distribución de sexo:')
display(df_adultos[SEX_VAR].value_counts(dropna=False).sort_index())

print('Valores especiales detectados en continuas clave:')
for variable in [AGE_VAR, AGE_AT_DIAGNOSIS_VAR, CARE_VISITS_VAR, HOSPITALIZATION_VAR]:
    n_special = int(df_adultos[variable].isin(CONTINUOUS_SPECIAL_MISSING_CODES).sum())
    print(f' - {variable}: {n_special}')

print('Valores especiales detectados en binarias clave:')
for variable in [INSULIN_VAR, DEPRESSION_VAR, *COMPLICATION_VARS.values()]:
    n_special = int(df_adultos[variable].isin(BINARY_SPECIAL_MISSING_CODES).sum())
    print(f' - {variable}: {n_special}')


Distribución de diagnóstico de diabetes (a0301):


,count
a0301,
1.0,1544
2.0,18
3.0,10351


Distribución de sexo:


,count
sexo,
1.0,4724
2.0,7189


Valores especiales detectados en continuas clave:
 - edad: 12
 - a0302: 5
 - a0303num: 1
 - a03061b: 0
Valores especiales detectados en binarias clave:
 - a0310: 0
 - a0202: 0
 - a0316a: 0
 - a0316b: 0
 - a0316c: 0
 - a0316d: 0
 - a0316e: 0
 - a0316f: 0
 - a0316g: 0
 - a0316h: 0
 - a0316i: 0
 - a0316j: 0


## 5. Seleccionar personas elegibles

La cohorte incluye solo adultos con `a0301 = 1`. Diabetes gestacional (`a0301 = 2`) y respuesta negativa (`a0301 = 3`) no entran a la cohorte.

In [5]:
df_source = df_adultos[REQUIRED_COLUMNS].copy()
df_source[IDENTIFIER] = df_source[IDENTIFIER].astype('string')

for variable in REQUIRED_COLUMNS:
    if variable != IDENTIFIER:
        df_source[variable] = pd.to_numeric(df_source[variable], errors='coerce')

flow_rows = [
    {'paso': 'Registros en Adultos', 'n_registros': int(len(df_source))},
    {
        'paso': 'Diabetes gestacional excluida (a0301 = 2)',
        'n_registros': int((df_source[DIABETES_VAR] == 2).sum())
    },
    {
        'paso': 'Sin diagnóstico de diabetes (a0301 = 3)',
        'n_registros': int((df_source[DIABETES_VAR] == 3).sum())
    }
]

df_cohort_raw = df_source.loc[
    df_source[DIABETES_VAR].isin(DIABETES_POSITIVE_VALUES)
].copy()

flow_rows.append({
    'paso': 'Cohorte con diabetes diagnosticada (a0301 = 1)',
    'n_registros': int(len(df_cohort_raw))
})

if df_cohort_raw.empty:
    raise ValueError('La cohorte quedó vacía. Verifica la codificación de a0301.')

df_flow = pd.DataFrame(flow_rows)
display(df_flow)


,paso,n_registros
0,Registros en Adultos,11913
1,Diabetes gestacional excluida (a0301 = 2),18
2,Sin diagnóstico de diabetes (a0301 = 3),10351
3,Cohorte con diabetes diagnosticada (a0301 = 1),1544


## 6. Recodificación de faltantes y respuestas binarias

Los códigos 89 y 99 se convierten a faltantes solo en variables continuas. El código 9 se convierte a faltante en variables binarias. Las respuestas positivas se derivan exclusivamente de `Sí = 1`; una respuesta `No = 2` y un faltante reciben valor 0 en los indicadores de antecedentes, con una auditoría separada de faltantes.

In [6]:
df = df_cohort_raw.copy()

continuous_variables = [AGE_VAR, AGE_AT_DIAGNOSIS_VAR, CARE_VISITS_VAR, HOSPITALIZATION_VAR]
binary_variables = [INSULIN_VAR, DEPRESSION_VAR, *COMPLICATION_VARS.values()]

special_codes_report = []

for variable in continuous_variables:
    n_special = int(df[variable].isin(CONTINUOUS_SPECIAL_MISSING_CODES).sum())
    df.loc[df[variable].isin(CONTINUOUS_SPECIAL_MISSING_CODES), variable] = np.nan
    special_codes_report.append({
        'variable': variable,
        'tipo': 'continua',
        'codigos_especiales': '89,99',
        'n_convertidos_a_nan': n_special
    })

for variable in binary_variables:
    n_special = int(df[variable].isin(BINARY_SPECIAL_MISSING_CODES).sum())
    df.loc[df[variable].isin(BINARY_SPECIAL_MISSING_CODES), variable] = np.nan
    special_codes_report.append({
        'variable': variable,
        'tipo': 'binaria',
        'codigos_especiales': '9',
        'n_convertidos_a_nan': n_special
    })

df_special_codes = pd.DataFrame(special_codes_report)
display(df_special_codes)

def derive_yes_indicator(series):
    return series.isin(BINARY_YES_VALUES).astype('int8')

df['insulina_diaria'] = derive_yes_indicator(df[INSULIN_VAR])
df['depresion'] = derive_yes_indicator(df[DEPRESSION_VAR])


,variable,tipo,codigos_especiales,n_convertidos_a_nan
0,edad,continua,"89,99",2
1,a0302,continua,"89,99",5
2,a0303num,continua,"89,99",1
3,a03061b,continua,"89,99",0
4,a0310,binaria,9,0
5,a0202,binaria,9,0
6,a0316a,binaria,9,0
7,a0316b,binaria,9,0
8,a0316c,binaria,9,0
9,a0316d,binaria,9,0


## 7. Crear variables derivadas

Las complicaciones se agrupan para evitar doble conteo de manifestaciones relacionadas. `anios_con_diabetes` se calcula como `edad - a0302`; valores negativos o mayores a la edad se tratan como inválidos y posteriormente se imputan con la mediana de la cohorte.

In [7]:
df['complicacion_pie'] = (
    derive_yes_indicator(df[COMPLICATION_VARS['ulceras_piernas_pies']])
    | derive_yes_indicator(df[COMPLICATION_VARS['amputacion']])
    | derive_yes_indicator(df[COMPLICATION_VARS['pie_diabetico']])
).astype('int8')

df['complicacion_visual'] = (
    derive_yes_indicator(df[COMPLICATION_VARS['disminucion_vision']])
    | derive_yes_indicator(df[COMPLICATION_VARS['perdida_vista']])
).astype('int8')

df['complicacion_renal'] = derive_yes_indicator(df[COMPLICATION_VARS['dialisis']])

df['complicacion_macrovascular'] = (
    derive_yes_indicator(df[COMPLICATION_VARS['infarto_cardiaco']])
    | derive_yes_indicator(df[COMPLICATION_VARS['infarto_cerebral']])
).astype('int8')

df['evento_agudo_diabetes'] = (
    derive_yes_indicator(df[COMPLICATION_VARS['coma_diabetico']])
    | derive_yes_indicator(df[COMPLICATION_VARS['hipoglucemia_grave']])
).astype('int8')

complication_columns = [
    'complicacion_pie',
    'complicacion_visual',
    'complicacion_renal',
    'complicacion_macrovascular',
    'evento_agudo_diabetes'
]

df['num_complicaciones'] = df[complication_columns].sum(axis=1).astype('int8')

df['anios_con_diabetes'] = df[AGE_VAR] - df[AGE_AT_DIAGNOSIS_VAR]
invalid_duration = (
    df['anios_con_diabetes'].lt(0)
    | df['anios_con_diabetes'].gt(df[AGE_VAR])
)
df.loc[invalid_duration, 'anios_con_diabetes'] = np.nan

df['consultas_control_12m'] = df[CARE_VISITS_VAR].copy()
df['num_hospitalizaciones'] = df[HOSPITALIZATION_VAR].copy()

for variable in ['consultas_control_12m', 'num_hospitalizaciones']:
    df.loc[df[variable].lt(0), variable] = np.nan

print('Duraciones inválidas transformadas a NaN:', int(invalid_duration.sum()))
display(df[[AGE_VAR, AGE_AT_DIAGNOSIS_VAR, 'anios_con_diabetes']].describe())
display(df[complication_columns + ['num_complicaciones']].describe())


Duraciones inválidas transformadas a NaN: 3


,edad,a0302,anios_con_diabetes
count,1542.000000,1539.000000,1534.000000
mean,59.201038,44.922027,14.317471
std,12.247281,14.744963,13.893791
min,21.000000,0.000000,0.000000
25%,51.000000,37.000000,4.000000
50%,60.000000,46.000000,10.000000
75%,67.750000,55.000000,20.000000
max,91.000000,86.000000,78.000000


,complicacion_pie,complicacion_visual,complicacion_renal,complicacion_macrovascular,evento_agudo_diabetes,num_complicaciones
count,1544.000000,1544.000000,1544.000000,1544.000000,1544.000000,1544.000000
mean,0.093912,0.502591,0.015544,0.034326,0.144430,0.790803
std,0.291801,0.500155,0.123743,0.182125,0.351639,0.836400
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
75%,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000


## 8. Control de calidad e imputación documentada

Se descartan registros sin clave, edad o sexo válido. `anios_con_diabetes`, consultas de control y hospitalizaciones se imputan por mediana dentro de la cohorte. Las variables binarias derivadas quedan en 0/1; los faltantes originales se registran en el reporte de códigos especiales.

In [8]:
valid_sex_mask = df[SEX_VAR].isin(SEX_CODES.keys())
core_missing_mask = df[IDENTIFIER].isna() | df[AGE_VAR].isna() | ~valid_sex_mask
n_removed_core = int(core_missing_mask.sum())
df = df.loc[~core_missing_mask].copy()

if df.empty:
    raise ValueError('No quedaron registros después de validar identificador, edad y sexo.')

imputation_rows = []
for variable in ['anios_con_diabetes', 'consultas_control_12m', 'num_hospitalizaciones']:
    n_missing_before = int(df[variable].isna().sum())
    median_value = float(df[variable].median()) if df[variable].notna().any() else 0.0
    df[variable] = df[variable].fillna(median_value)
    imputation_rows.append({
        'variable': variable,
        'estrategia': 'imputacion_mediana_cohorte',
        'faltantes_antes': n_missing_before,
        'valor_imputado': median_value,
        'faltantes_despues': int(df[variable].isna().sum())
    })

df_imputation = pd.DataFrame(imputation_rows)
print('Registros removidos por clave/edad/sexo inválido:', n_removed_core)
display(df_imputation)


Registros removidos por clave/edad/sexo inválido: 2


,variable,estrategia,faltantes_antes,valor_imputado,faltantes_despues
0,anios_con_diabetes,imputacion_mediana_cohorte,8,10.0,0
1,consultas_control_12m,imputacion_mediana_cohorte,1,6.0,0
2,num_hospitalizaciones,imputacion_mediana_cohorte,0,0.0,0


## 9. Score de riesgo de simulación

El score es una regla transparente para estratificar perfiles en el simulador. No equivale a una regla de predicción clínica validada.

In [9]:
df['score_riesgo'] = (
    2 * df['num_complicaciones']
    + 2 * (df['num_hospitalizaciones'] > 0).astype('int8')
    + df['insulina_diaria']
    + df['depresion']
    + (df[AGE_VAR] >= 60).astype('int8')
    + (df['anios_con_diabetes'] >= 10).astype('int8')
).astype('int16')

df['categoria_riesgo'] = pd.cut(
    df['score_riesgo'],
    bins=[-np.inf, 1, 3, np.inf],
    labels=['bajo', 'medio', 'alto']
).astype('string')

display(
    df['categoria_riesgo']
    .value_counts(dropna=False)
    .rename_axis('categoria_riesgo')
    .reset_index(name='n_personas')
)
display(df['score_riesgo'].describe())


,categoria_riesgo,n_personas
0,alto,612
1,medio,544
2,bajo,386


,score_riesgo
count,1542.000000
mean,3.164073
std,2.337446
min,0.000000
25%,1.250000
50%,3.000000
75%,4.000000
max,12.000000


## 10. Cohorte final y validaciones

Se preservan variables necesarias para el estado basal, auditoría de equidad y trazabilidad del score. La cohorte debe ser totalmente numérica en sus variables modelables y no contener valores no finitos.

In [10]:
FINAL_COLUMNS = [
    IDENTIFIER, AGE_VAR, SEX_VAR, 'anios_con_diabetes',
    'insulina_diaria', 'depresion', 'consultas_control_12m',
    'num_hospitalizaciones',
    'complicacion_pie', 'complicacion_visual', 'complicacion_renal',
    'complicacion_macrovascular', 'evento_agudo_diabetes',
    'num_complicaciones', 'score_riesgo', 'categoria_riesgo'
]

df_cohorte_diabetes = df[FINAL_COLUMNS].copy()

numeric_columns = [
    AGE_VAR, SEX_VAR, 'anios_con_diabetes', 'insulina_diaria', 'depresion',
    'consultas_control_12m', 'num_hospitalizaciones',
    *complication_columns, 'num_complicaciones', 'score_riesgo'
]

assert df_cohorte_diabetes[IDENTIFIER].notna().all(), 'Hay FOLIO_INT faltantes.'
assert not df_cohorte_diabetes[IDENTIFIER].duplicated().any(), 'Hay FOLIO_INT duplicados.'
assert df_cohorte_diabetes[SEX_VAR].isin(SEX_CODES.keys()).all(), 'Sexo fuera de codificación esperada.'
assert np.isfinite(df_cohorte_diabetes[numeric_columns].to_numpy(dtype=float)).all(), 'Hay NaN o infinitos en columnas numéricas.'
assert df_cohorte_diabetes['anios_con_diabetes'].between(0, df_cohorte_diabetes[AGE_VAR]).all(), 'Años con diabetes fuera de rango.'
assert df_cohorte_diabetes['num_complicaciones'].between(0, 5).all(), 'Número de complicaciones fuera de rango.'
assert df_cohorte_diabetes['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all(), 'Categoría de riesgo inválida.'

print('Cohorte validada correctamente.')
print('Dimensiones:', df_cohorte_diabetes.shape)
display(df_cohorte_diabetes.head())
display(df_cohorte_diabetes.dtypes.rename('tipo'))


Cohorte validada correctamente.
Dimensiones: (1542, 16)


,FOLIO_INT,edad,sexo,anios_con_diabetes,insulina_diaria,depresion,consultas_control_12m,num_hospitalizaciones,complicacion_pie,complicacion_visual,complicacion_renal,complicacion_macrovascular,evento_agudo_diabetes,num_complicaciones,score_riesgo,categoria_riesgo
9,2022_01001011_01,66.0,1.0,0.0,0,0,0.0,0.0,0,0,0,0,0,0,1,bajo
15,2022_01001015_01,65.0,2.0,20.0,1,1,0.0,0.0,0,1,0,0,0,1,6,alto
17,2022_01001017_02,70.0,2.0,8.0,0,1,0.0,0.0,0,1,0,0,0,1,4,alto
20,2022_01001018_02,65.0,1.0,40.0,0,0,0.0,0.0,0,1,1,0,0,2,6,alto
30,2022_01001027_01,76.0,1.0,56.0,0,0,0.0,0.0,0,0,0,0,0,0,2,medio


,tipo
FOLIO_INT,string[python]
edad,float64
sexo,float64
anios_con_diabetes,float64
insulina_diaria,int8
depresion,int8
consultas_control_12m,float64
num_hospitalizaciones,float64
complicacion_pie,int8
complicacion_visual,int8


## 11. Exportar cohorte, diccionario y bitácoras

La cohorte canónica se guarda solo en la ruta persistente de Drive. Los notebooks posteriores deben buscarla desde `data/processed/cohorte_diabetes.csv`.

In [11]:
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'
FLOW_PATH = METADATA_DIR / 'flujo_preparacion_cohorte.csv'
IMPUTATION_PATH = METADATA_DIR / 'reporte_imputacion_cohorte.csv'
SPECIAL_CODES_PATH = METADATA_DIR / 'reporte_codigos_especiales_cohorte.csv'
DICTIONARY_PATH = METADATA_DIR / 'diccionario_cohorte.csv'
SPEC_PATH = SPECIFICATIONS_DIR / 'cohort_specification_v1.json'

df_cohorte_diabetes.to_csv(COHORT_PATH, index=False, encoding='utf-8-sig')
df_flow.to_csv(FLOW_PATH, index=False, encoding='utf-8-sig')
df_imputation.to_csv(IMPUTATION_PATH, index=False, encoding='utf-8-sig')
df_special_codes.to_csv(SPECIAL_CODES_PATH, index=False, encoding='utf-8-sig')

dictionary_rows = [
    {'variable': IDENTIFIER, 'descripcion': 'Folio único de integrante', 'tipo': 'identificador', 'uso': 'llave técnica'},
    {'variable': AGE_VAR, 'descripcion': 'Edad actual', 'tipo': 'numérica', 'uso': 'estado y equidad'},
    {'variable': SEX_VAR, 'descripcion': 'Sexo ENSANUT: 1=hombre, 2=mujer', 'tipo': 'categórica', 'uso': 'estado y equidad'},
    {'variable': 'anios_con_diabetes', 'descripcion': 'Edad actual menos edad al diagnóstico (edad - a0302)', 'tipo': 'numérica derivada', 'uso': 'estado y riesgo'},
    {'variable': 'insulina_diaria', 'descripcion': 'Indicador: a0310 = Sí', 'tipo': 'binaria derivada', 'uso': 'estado y riesgo'},
    {'variable': 'depresion', 'descripcion': 'Indicador: a0202 = Sí', 'tipo': 'binaria derivada', 'uso': 'estado y riesgo'},
    {'variable': 'consultas_control_12m', 'descripcion': 'Consultas de control de diabetes en 12 meses (a0303num)', 'tipo': 'numérica', 'uso': 'estado'},
    {'variable': 'num_hospitalizaciones', 'descripcion': 'Hospitalizaciones por diabetes en 12 meses (a03061b)', 'tipo': 'numérica', 'uso': 'riesgo y evaluación'},
    {'variable': 'complicacion_pie', 'descripcion': 'Úlcera, amputación o pie diabético', 'tipo': 'binaria derivada', 'uso': 'riesgo'},
    {'variable': 'complicacion_visual', 'descripcion': 'Disminución visual o pérdida de vista por diabetes', 'tipo': 'binaria derivada', 'uso': 'riesgo'},
    {'variable': 'complicacion_renal', 'descripcion': 'Diálisis por diabetes', 'tipo': 'binaria derivada', 'uso': 'riesgo'},
    {'variable': 'complicacion_macrovascular', 'descripcion': 'Infarto cardiaco o cerebral por diabetes', 'tipo': 'binaria derivada', 'uso': 'riesgo'},
    {'variable': 'evento_agudo_diabetes', 'descripcion': 'Coma diabético o hipoglucemia grave', 'tipo': 'binaria derivada', 'uso': 'riesgo'},
    {'variable': 'num_complicaciones', 'descripcion': 'Suma de cinco categorías clínicas no solapadas', 'tipo': 'numérica derivada', 'uso': 'estado y riesgo'},
    {'variable': 'score_riesgo', 'descripcion': 'Regla transparente de simulación; no es escala clínica validada', 'tipo': 'numérica derivada', 'uso': 'inicialización del MDP'},
    {'variable': 'categoria_riesgo', 'descripcion': 'Bajo, medio o alto según score de simulación', 'tipo': 'categórica derivada', 'uso': 'estado, evaluación y equidad'}
]

pd.DataFrame(dictionary_rows).to_csv(DICTIONARY_PATH, index=False, encoding='utf-8-sig')

cohort_specification = {
    'spec_version': '2.0',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_file': str(ADULTOS_PATH) if ADULTOS_PATH else 'df_adultos_en_memoria',
    'identifier': IDENTIFIER,
    'inclusion_rule': 'a0301 = 1',
    'exclusion_rule_gestational_diabetes': 'a0301 = 2',
    'exclusion_rule_no_diabetes': 'a0301 = 3',
    'binary_coding': {'si': 1, 'no': 2, 'especial_no_sabe_no_responde': 9},
    'continuous_special_missing_codes': [89, 99],
    'sex_coding': {'hombre': 1, 'mujer': 2},
    'duration_formula': 'anios_con_diabetes = edad - a0302',
    'risk_score_note': 'Score transparente para simulación; no validado clínicamente.'
}

with SPEC_PATH.open('w', encoding='utf-8') as file:
    json.dump(cohort_specification, file, ensure_ascii=False, indent=2)

print('Archivos exportados:')
for path in [COHORT_PATH, FLOW_PATH, IMPUTATION_PATH, SPECIAL_CODES_PATH, DICTIONARY_PATH, SPEC_PATH]:
    print(' -', path, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)


Archivos exportados:
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/processed/cohorte_diabetes.csv | existe: True | bytes: 96346
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/flujo_preparacion_cohorte.csv | existe: True | bytes: 191
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/reporte_imputacion_cohorte.csv | existe: True | bytes: 241
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/reporte_codigos_especiales_cohorte.csv | existe: True | bytes: 386
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/diccionario_cohorte.csv | existe: True | bytes: 1421
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/specifications/cohort_specification_v1.json | existe: True | bytes: 692


## Resultado esperado

El notebook debe crear `data/processed/cohorte_diabetes.csv` y los archivos de auditoría en `data/metadata/`. Después de confirmar que todas las validaciones pasaron, continúa con `04_definicion_mdp.ipynb`.